# Retail Basket Value Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `basket_value`

## 2 · Project Overview

We predict **shopping basket value** based on item count, average item price, customer demographics (age, loyalty), channel, store type, timing (day, hour), coupon usage, and payment method.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given a shopping transaction's item count, average item price, customer loyalty, channel, store type, timing, and coupon usage, predict the total basket value.

## 5 · Why This Project Matters

- **Basket value prediction** drives revenue forecasting and inventory planning.
- Understanding basket drivers enables upselling and cross-selling strategies.
- Coupon optimization requires knowing their impact on basket size.
- Demonstrates regression with multiplicative pricing components.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | num_items, avg_item_price, customer_age, customer_loyalty_years, channel, store_type, day_of_week, is_weekend, hour_of_day, has_coupon, coupon_discount_pct, payment_method |
| **Target** | basket_value (continuous, USD) |
| **Range** | ~$3 – $2,000 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "basket_value"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

Synthetic dataset: 8,000 retail shopping basket records with customer and transaction features.

In [ ]:
np.random.seed(SEED)
n = 8000
num_items = np.random.poisson(8, n).clip(1, 50)
avg_item_price = np.round(np.random.lognormal(2.5, 0.6, n).clip(2, 200), 2)
customer_age = np.random.randint(18, 75, n)
customer_loyalty_years = np.round(np.random.exponential(3, n).clip(0, 20), 1)
channel = np.random.choice(["In-Store", "Online", "Mobile App"], n, p=[0.4, 0.35, 0.25])
chan_mult = {"In-Store": 1.1, "Online": 1.0, "Mobile App": 0.9}
chan_val = np.array([chan_mult[c] for c in channel])

store_type = np.random.choice(["Supermarket", "Department", "Specialty", "Convenience"], n,
                               p=[0.35, 0.25, 0.2, 0.2])
store_base = {"Supermarket": 1.0, "Department": 1.3, "Specialty": 1.5, "Convenience": 0.7}
store_val = np.array([store_base[s] for s in store_type])

day_of_week = np.random.randint(0, 7, n)
is_weekend = (day_of_week >= 5).astype(int)
hour_of_day = np.random.randint(8, 22, n)
has_coupon = np.random.binomial(1, 0.3, n)
coupon_discount_pct = has_coupon * np.random.uniform(5, 25, n)
payment_method = np.random.choice(["Cash", "Credit Card", "Debit Card", "Digital Wallet"], n,
                                   p=[0.15, 0.4, 0.3, 0.15])

# Basket value model
basket_value = (num_items * avg_item_price * store_val * chan_val
                * (1 + 0.1 * is_weekend)
                * (1 + 0.005 * customer_loyalty_years)
                * (1 - coupon_discount_pct / 200))
basket_value = np.round(basket_value + np.random.normal(0, 10, n), 2).clip(3, 2000)

df = pd.DataFrame({
    "num_items": num_items, "avg_item_price": avg_item_price, "customer_age": customer_age,
    "customer_loyalty_years": customer_loyalty_years, "channel": channel,
    "store_type": store_type, "day_of_week": day_of_week, "is_weekend": is_weekend,
    "hour_of_day": hour_of_day, "has_coupon": has_coupon,
    "coupon_discount_pct": coupon_discount_pct, "payment_method": payment_method,
    "basket_value": basket_value
})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['basket_value'].describe()}")
df.head()

## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["num_items"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("Number of Items"); axes[0][0].set_ylabel("Basket Value ($)")
axes[0][0].set_title("Items vs Basket Value")

axes[0][1].scatter(df["avg_item_price"], df[TARGET], alpha=0.2, s=10)
axes[0][1].set_xlabel("Avg Item Price ($)"); axes[0][1].set_ylabel("Basket Value ($)")
axes[0][1].set_title("Avg Price vs Basket Value")

axes[0][2].scatter(df["customer_loyalty_years"], df[TARGET], alpha=0.2, s=10)
axes[0][2].set_xlabel("Loyalty Years"); axes[0][2].set_ylabel("Basket Value ($)")
axes[0][2].set_title("Loyalty vs Basket Value")

df.boxplot(column=TARGET, by="store_type", ax=axes[1][0])
axes[1][0].set_title("Basket Value by Store Type")
axes[1][0].tick_params(axis="x", rotation=45)

df.boxplot(column=TARGET, by="channel", ax=axes[1][1])
axes[1][1].set_title("Basket Value by Channel")

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1][2], square=True, cbar_kws={"shrink": 0.8})
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `basket_value`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Basket Value ($)")

df.boxplot(column=TARGET, by="store_type", ax=axes[1])
axes[1].set_title("Basket Value by Store Type")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()
print(f"Range: ${df[TARGET].min():,.2f} to ${df[TARGET].max():,.2f}")
print(f"Mean: ${df[TARGET].mean():,.2f}, Median: ${df[TARGET].median():,.2f}")
print(f"Skew: {df[TARGET].skew():.2f}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `channel`, `store_type`, `payment_method` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: High basket values from luxury/specialty stores — keep them.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["items_x_price"] = X_train["num_items"] * X_train["avg_item_price"]
X_test["items_x_price"] = X_test["num_items"] * X_test["avg_item_price"]

X_train["effective_discount"] = X_train["coupon_discount_pct"] * X_train["has_coupon"]
X_test["effective_discount"] = X_test["coupon_discount_pct"] * X_test["has_coupon"]

X_train["hour_sin"] = np.sin(2 * np.pi * X_train["hour_of_day"] / 24)
X_test["hour_sin"] = np.sin(2 * np.pi * X_test["hour_of_day"] / 24)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Items × price** is the dominant basket value driver (math identity with noise).
- **Store type** (Specialty: 1.5× > Department: 1.3× > Supermarket: 1.0×) sets pricing tier.
- **In-store** channel generates ~10% higher baskets than mobile.
- **Weekend shopping** increases basket value by ~10%.
- **Customer loyalty** adds incremental value (~0.5%/year).
- **Coupons** reduce basket value but may increase overall volume.

**Business takeaway:** Drive basket value through upselling (more items) and premiumization (higher avg price). Coupons trade margin for traffic.

## 26 · Limitations

1. Synthetic data with simplified basket dynamics.
2. No product category breakdown.
3. Missing promotional/seasonal effects.
4. No customer purchase history.
5. Simplified coupon mechanics.

## 27 · How to Improve This Project

1. Use real POS transaction data.
2. Add product category features.
3. Include promotional calendars.
4. Model basket composition (category mix).
5. Add customer RFM (recency, frequency, monetary) features.

## 28 · Production Considerations

- Deploy for real-time basket value prediction.
- Use for personalized coupon/offer targeting.
- Monitor basket trends for inventory planning.
- Update weekly with new transaction data.
- Output basket value distributions for revenue forecasting.

## 29 · Common Mistakes

1. Not creating `items × price` interaction (the core basket identity).
2. Ignoring store type pricing differences.
3. Treating coupons as always negative (they drive traffic).
4. Not separating channel effects.
5. Using raw hour instead of cyclical encoding.

## 30 · Mini Challenge / Exercises

1. Log-transform `basket_value` — does R² improve for Linear Regression?
2. Remove `has_coupon` and `coupon_discount_pct` — how much does RMSE change?
3. Plot basket value by hour_of_day — identify shopping patterns.
4. Create `price_tier = pd.qcut(avg_item_price, 3)` and analyze.
5. Build separate models by store_type.

## 31 · Final Summary / Key Takeaways

1. **Items × price** is the dominant basket value predictor.
2. **Store type** sets the pricing tier (Specialty > Department > Supermarket).
3. **In-store** shopping generates higher baskets than mobile.
4. **Weekend** shopping increases basket value by ~10%.
5. **Coupons** trade margin for traffic volume.
6. **Customer loyalty** adds small but consistent basket increases.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))